In [4]:
import os
import pandas as pd
import geopandas as gpd
from rasterstats import zonal_stats
import glob
import rasterio

# 1. LOAD AND CHECK BOUNDARIES
boundary_path = "../data/mp_boundaries.json"
districts = gpd.read_file(boundary_path)

print(f"DEBUG: Districts loaded: {len(districts)}")
if len(districts) == 0:
    print("❌ ERROR: Your mp_boundaries.json is empty! Go back to the 'Getting MP Boundaries' notebook and re-run it.")

# 2. DEFINE DATA DIRECTORIES
raw_path = "../data/raw"
data_sources = {
    'LST': 'LST_2018/raster',
    'SSM': 'SSM_2018/raster',
    'NDVI': 'NDVI_2018_1/raster'
}

def extract_time_series(folder_key, folder_relative_path):
    all_records = []
    search_path = os.path.join(raw_path, folder_relative_path, "*.tif")
    tif_files = sorted(glob.glob(search_path))
    
    if not tif_files:
        return pd.DataFrame()

    print(f"Processing {len(tif_files)} files for {folder_key}...")
    
    # Identify the district name column once
    name_col = 'district_name' if 'district_name' in districts.columns else districts.columns[0]
    
    for tif_path in tif_files:
        file_name = os.path.basename(tif_path)
        date_str = file_name.replace('.tif', '').replace(f"{folder_key}_", "").replace("SSM_", "").replace("NDVI_", "")

        with rasterio.open(tif_path) as src:
            tif_crs = src.crs
            # Re-project map to match the image
            districts_proj = districts.to_crs(tif_crs)

        # We add 'all_touched=True' to ensure small districts aren't skipped
        stats = zonal_stats(
            districts_proj.__geo_interface__['features'], 
            tif_path, 
            stats="mean",
            all_touched=True
        )
        
        for i, s in enumerate(stats):
            if s['mean'] is not None:
                all_records.append({
                    'Date': date_str,
                    'District': districts.iloc[i][name_col],
                    folder_key: s['mean']
                })
            
    df = pd.DataFrame(all_records)
    print(f"✅ Extracted {len(df)} records for {folder_key}")
    return df

# 3. RUN EXTRACTION
print("Starting Debug Extraction...")

lst_df = extract_time_series('LST', data_sources['LST'])
ssm_df = extract_time_series('SSM', data_sources['SSM'])
ndvi_df = extract_time_series('NDVI', data_sources['NDVI'])

# 4. MERGE DATASETS
if not lst_df.empty or not ssm_df.empty:
    print("Merging DataFrames...")
    # Start with the one that has data
    master_df = lst_df if not lst_df.empty else ssm_df
    
    if not ssm_df.empty and master_df is not lst_df:
        master_df = pd.merge(master_df, ssm_df, on=['Date', 'District'], how='outer')
    elif not ssm_df.empty:
        master_df = pd.merge(master_df, ssm_df, on=['Date', 'District'], how='outer')
        
    if not ndvi_df.empty:
        master_df = pd.merge(master_df, ndvi_df, on=['Date', 'District'], how='outer')

    # 5. CLEAN UP
    master_df['Date'] = pd.to_datetime(master_df['Date'], errors='coerce')
    master_df = master_df.sort_values(['District', 'Date']).dropna(subset=['Date'])
    
    os.makedirs("../data/processed", exist_ok=True)
    master_df.to_csv("../data/processed/real_2018_master.csv", index=False)
    
    print(f"\n✅ FINAL SUCCESS: Table has {len(master_df)} rows.")
    display(master_df.head(10))
else:
    print("❌ CRITICAL FAILURE: No data was extracted from any file.")
    print("Possible Reason 1: The images and the map don't overlap geographically.")
    print("Possible Reason 2: The .tif files might be empty/corrupt.") 

DEBUG: Districts loaded: 0
❌ ERROR: Your mp_boundaries.json is empty! Go back to the 'Getting MP Boundaries' notebook and re-run it.
Starting Debug Extraction...
Processing 12 files for LST...
✅ Extracted 0 records for LST
Processing 11 files for SSM...
✅ Extracted 0 records for SSM
Processing 12 files for NDVI...
✅ Extracted 0 records for NDVI
❌ CRITICAL FAILURE: No data was extracted from any file.
Possible Reason 1: The images and the map don't overlap geographically.
Possible Reason 2: The .tif files might be empty/corrupt.
